# Task 15 — Corrected 61D versus 64D comparison

Compare the selected 61-feature stability model from Task 12 with the same model plus `in_TMD`, `dist_to_nearest_TMD`, and `delta_membrane_insertion`. Task 12 OOF predictions and fold assignments are reused as the fixed 61D comparator. The 64D model uses exactly the same grouped folds.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")

BASE = Path("/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial")
FEATURES_CSV = BASE / "features_v3.csv"
TASK12_OOF_CSV = BASE / "task12_ddg_ablation_oof.csv"
TMD_CSV = BASE / "tmd_features.csv"
OOF_CSV = BASE / "task15_full64_oof.csv"
METRICS_CSV = BASE / "task15_full64_metrics.csv"
IMPORTANCE_CSV = BASE / "task15_full64_feature_importance.csv"
RANDOM_STATE = 42
N_COMPONENTS = 50
ESM_COLS = [f"esm_{i}" for i in range(1280)]
STRUCT_COLS = ["plddt", "sasa", "rsa", "ss_helix", "ss_strand",
               "ss_coil", "delta_hydrophobicity"]
DDG_COLS = ["ddg_esm2", "ddg_struct", "ddg_rasp", "ddg_foldx"]
TMD_COLS = ["in_TMD", "dist_to_nearest_TMD", "delta_membrane_insertion"]
print("61D = 50 PCA + 7 structural + 4 stability")
print("64D = 61D + 3 TMD")

61D = 50 PCA + 7 structural + 4 stability
64D = 61D + 3 TMD


## 1. Canonical KEY alignment

In [2]:
df = pd.read_csv(FEATURES_CSV)
required = ["KEY", "Gene", "Mutation_used", "source", "Mislocalized"]
missing = [c for c in required + ESM_COLS + STRUCT_COLS if c not in df.columns]
assert not missing, f"Missing features_v3 columns: {missing[:10]}"
assert len(df) == 2179 and df["KEY"].is_unique
assert df["Mislocalized"].isin([0, 1]).all() and int(df["Mislocalized"].sum()) == 236

task12 = pd.read_csv(TASK12_OOF_CSV)
task12_cols = ["KEY", "fold", "oof_final_all_ddgs",
               "final_alphamissense_score", "alphamissense_status"]
assert all(c in task12.columns for c in task12_cols)
assert len(task12) == 2179 and task12["KEY"].is_unique
df = df.merge(task12[task12_cols], on="KEY", how="left", validate="one_to_one")

for name in DDG_COLS:
    feature_df = pd.read_csv(BASE / f"{name}.csv")
    assert feature_df["KEY"].is_unique and name in feature_df.columns
    assert not (set(feature_df["KEY"]) - set(df["KEY"]))
    df = df.merge(feature_df[["KEY", name]], on="KEY", how="left",
                  validate="one_to_one")

tmd = pd.read_csv(TMD_CSV)
assert tmd["KEY"].is_unique and all(c in tmd.columns for c in TMD_COLS)
assert set(tmd["KEY"]) == set(df["KEY"])
df = df.merge(tmd[["KEY"] + TMD_COLS], on="KEY", how="left",
              validate="one_to_one")
assert len(df) == 2179 and df["KEY"].is_unique
assert df["fold"].notna().all() and df["oof_final_all_ddgs"].between(0, 1).all()

for c in DDG_COLS + TMD_COLS:
    print(f"{c:28s}: {df[c].notna().sum()}/{len(df)} available")

ddg_esm2                    : 2179/2179 available
ddg_struct                  : 2168/2179 available
ddg_rasp                    : 2168/2179 available
ddg_foldx                   : 2166/2179 available
in_TMD                      : 2179/2179 available
dist_to_nearest_TMD         : 2179/2179 available
delta_membrane_insertion    : 2179/2179 available


## 2. Train 64D model on the fixed Task 12 folds

In [3]:
X_esm = df[ESM_COLS].to_numpy(dtype=np.float32)
X_struct = df[STRUCT_COLS].to_numpy(dtype=np.float32)
X_extra = {c: df[[c]].to_numpy(dtype=np.float32) for c in DDG_COLS + TMD_COLS}
y = df["Mislocalized"].astype(int).to_numpy()
groups = df["Gene"].astype(str).to_numpy()
fold_id = df["fold"].astype(int).to_numpy()
oof_61 = df["oof_final_all_ddgs"].to_numpy(dtype=np.float32)
oof_64 = np.full(len(df), np.nan, dtype=np.float32)
fold_metrics = []

def make_model():
    return XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.5,
        objective="binary:logistic", eval_metric="aucpr",
        random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
    )

for fold in sorted(np.unique(fold_id)):
    train_idx = np.flatnonzero(fold_id != fold)
    test_idx = np.flatnonzero(fold_id == fold)
    assert set(groups[train_idx]).isdisjoint(set(groups[test_idx]))

    imp_e, scl_e = SimpleImputer(strategy="median"), StandardScaler()
    e_train = scl_e.fit_transform(imp_e.fit_transform(X_esm[train_idx])).astype(np.float32)
    e_test = scl_e.transform(imp_e.transform(X_esm[test_idx])).astype(np.float32)
    pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
    parts_train = [pca.fit_transform(e_train).astype(np.float32)]
    parts_test = [pca.transform(e_test).astype(np.float32)]

    imp_s, scl_s = SimpleImputer(strategy="median"), StandardScaler()
    parts_train.append(scl_s.fit_transform(imp_s.fit_transform(X_struct[train_idx])).astype(np.float32))
    parts_test.append(scl_s.transform(imp_s.transform(X_struct[test_idx])).astype(np.float32))

    for name in DDG_COLS + TMD_COLS:
        imp = SimpleImputer(strategy="median")
        parts_train.append(imp.fit_transform(X_extra[name][train_idx]).astype(np.float32))
        parts_test.append(imp.transform(X_extra[name][test_idx]).astype(np.float32))

    X_train = np.hstack(parts_train).astype(np.float32)
    X_test = np.hstack(parts_test).astype(np.float32)
    assert X_train.shape[1] == 64
    model = make_model()
    weights = compute_sample_weight("balanced", y[train_idx])
    model.fit(X_train, y[train_idx], sample_weight=weights, verbose=False)
    oof_64[test_idx] = model.predict_proba(X_test)[:, 1]

    for model_name, pred in [("stability_61", oof_61[test_idx]),
                             ("stability_tmd_64", oof_64[test_idx])]:
        fold_metrics.append({"scope": "fold", "model": model_name, "fold": fold,
            "n": len(test_idx), "n_positive": int(y[test_idx].sum()),
            "prevalence": float(y[test_idx].mean()),
            "auroc": roc_auc_score(y[test_idx], pred),
            "auprc": average_precision_score(y[test_idx], pred)})
    print(f"Fold {fold}: 61D={roc_auc_score(y[test_idx], oof_61[test_idx]):.4f}, "
          f"64D={roc_auc_score(y[test_idx], oof_64[test_idx]):.4f}")

assert np.isfinite(oof_64).all() and np.logical_and(oof_64 >= 0, oof_64 <= 1).all()

Fold 0: 61D=0.6459, 64D=0.6608
Fold 1: 61D=0.5959, 64D=0.6314
Fold 2: 61D=0.5924, 64D=0.6483
Fold 3: 61D=0.6416, 64D=0.6344
Fold 4: 61D=0.6976, 64D=0.6743


## 3. Full and paired AlphaMissense comparison

In [4]:
def metrics(scope, name, yy, pp):
    return {"scope": scope, "model": name, "fold": np.nan, "n": len(yy),
            "n_positive": int(yy.sum()), "prevalence": float(yy.mean()),
            "auroc": roc_auc_score(yy, pp),
            "auprc": average_precision_score(yy, pp)}

full61 = metrics("full_oof", "stability_61", y, oof_61)
full64 = metrics("full_oof", "stability_tmd_64", y, oof_64)
print("Full-cohort OOF comparison")
print(f"n={len(y)}, positives={int(y.sum())}, prevalence={y.mean():.4f}")
print(f"  61D: AUROC={full61['auroc']:.4f}, AUPRC={full61['auprc']:.4f}")
print(f"  64D: AUROC={full64['auroc']:.4f}, AUPRC={full64['auprc']:.4f}")
print(f"  Delta: AUROC={full64['auroc']-full61['auroc']:+.4f}, "
      f"AUPRC={full64['auprc']-full61['auprc']:+.4f}")

paired = df["final_alphamissense_score"].notna().to_numpy()
yp = y[paired]
am = df.loc[paired, "final_alphamissense_score"].to_numpy(dtype=float)
paired_am = metrics("paired_alphamissense", "alphamissense", yp, am)
paired61 = metrics("paired_alphamissense", "stability_61", yp, oof_61[paired])
paired64 = metrics("paired_alphamissense", "stability_tmd_64", yp, oof_64[paired])
print("\nPaired AlphaMissense comparison")
print(f"n={len(yp)}, positives={int(yp.sum())}, prevalence={yp.mean():.4f}")
for r in [paired_am, paired61, paired64]:
    print(f"  {r['model']:18s}: AUROC={r['auroc']:.4f}, AUPRC={r['auprc']:.4f}")
print(f"  64D vs AM: AUROC={paired64['auroc']-paired_am['auroc']:+.4f}, "
      f"AUPRC={paired64['auprc']-paired_am['auprc']:+.4f}")

Full-cohort OOF comparison
n=2179, positives=236, prevalence=0.1083
  61D: AUROC=0.6286, AUPRC=0.1758
  64D: AUROC=0.6422, AUPRC=0.1981
  Delta: AUROC=+0.0135, AUPRC=+0.0223

Paired AlphaMissense comparison
n=2140, positives=235, prevalence=0.1098
  alphamissense     : AUROC=0.6491, AUPRC=0.1619
  stability_61      : AUROC=0.6311, AUPRC=0.1780
  stability_tmd_64  : AUROC=0.6442, AUPRC=0.1999
  64D vs AM: AUROC=-0.0048, AUPRC=+0.0380


## 4. Save OOF predictions and metrics

In [5]:
out = df[["KEY", "Gene", "Mutation_used", "source", "Mislocalized",
          "fold", "final_alphamissense_score", "alphamissense_status"]].copy()
out["oof_stability_61"] = oof_61
out["oof_stability_tmd_64"] = oof_64
out["paired_alphamissense"] = paired
assert len(out) == 2179 and out["KEY"].is_unique
out.to_csv(OOF_CSV, index=False)
pd.DataFrame(fold_metrics + [full61, full64, paired_am, paired61, paired64]).to_csv(
    METRICS_CSV, index=False)
print(f"Saved: {OOF_CSV}")
print(f"Saved: {METRICS_CSV}")

Saved: /mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/task15_full64_oof.csv
Saved: /mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/task15_full64_metrics.csv


## 5. Descriptive 64D feature importance

The full-data fit below is used only for feature ranking, not performance estimation.

In [6]:
imp_e, scl_e = SimpleImputer(strategy="median"), StandardScaler()
e_all = scl_e.fit_transform(imp_e.fit_transform(X_esm)).astype(np.float32)
pca = PCA(n_components=N_COMPONENTS, random_state=RANDOM_STATE)
parts = [pca.fit_transform(e_all).astype(np.float32)]
imp_s, scl_s = SimpleImputer(strategy="median"), StandardScaler()
parts.append(scl_s.fit_transform(imp_s.fit_transform(X_struct)).astype(np.float32))
for name in DDG_COLS + TMD_COLS:
    parts.append(SimpleImputer(strategy="median").fit_transform(X_extra[name]).astype(np.float32))
X_all = np.hstack(parts).astype(np.float32)
assert X_all.shape == (2179, 64)
model = make_model()
model.fit(X_all, y, sample_weight=compute_sample_weight("balanced", y), verbose=False)
names = [f"PC{i+1}" for i in range(50)] + STRUCT_COLS + DDG_COLS + TMD_COLS
importance = pd.DataFrame({"feature": names, "importance": model.feature_importances_})
importance = importance.sort_values("importance", ascending=False).reset_index(drop=True)
importance["rank"] = np.arange(1, len(importance)+1)
importance.to_csv(IMPORTANCE_CSV, index=False)
print(importance.head(20).to_string(index=False))
print("\nDDG and TMD ranks:")
print(importance[importance["feature"].isin(DDG_COLS + TMD_COLS)].to_string(index=False))

                 feature  importance  rank
                  in_TMD    0.036551     1
                ddg_esm2    0.036359     2
     dist_to_nearest_TMD    0.029027     3
delta_membrane_insertion    0.026835     4
                     PC9    0.020571     5
                     PC1    0.020370     6
                ddg_rasp    0.019963     7
                    PC36    0.019711     8
                    PC39    0.018417     9
                    PC33    0.018253    10
                    PC37    0.017914    11
                    PC42    0.017469    12
                     PC5    0.017252    13
                   plddt    0.016981    14
                 ss_coil    0.016218    15
                    PC13    0.016086    16
                    PC29    0.016010    17
                    PC41    0.016004    18
                    PC30    0.015946    19
                    PC25    0.015881    20

DDG and TMD ranks:
                 feature  importance  rank
                  in_TMD    0.0365